In [4]:
!pip install syft -q

In [1]:
import syft as sy

server_a = sy.Worker(name="client_a", dev_mode=True)
server_b = sy.Worker(name="client_b", dev_mode=True)
client_a = server_a.root_client
client_b = server_b.root_client
print(client_a)
print(client_b)

<DatasiteClient: client_a>
<DatasiteClient: client_b>


In [2]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import Subset, DataLoader
import syft as sy

transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
full_train = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_data  = datasets.MNIST('./data', train=False, download=True, transform=transform)

def to_flat_tensors(subset):
    loader = DataLoader(subset, batch_size=len(subset), shuffle=False, num_workers=0)
    X, y = next(iter(loader))
    return X.view(-1, 784).float(), y.long()

X_a, y_a = to_flat_tensors(Subset(full_train, range(0, 30000)))
X_b, y_b = to_flat_tensors(Subset(full_train, range(30000, 60000)))

dataset_a = sy.Dataset(name="mnist_a", asset_list=[
    sy.Asset(name="X", data=X_a, mock=torch.zeros(10, 784)),
    sy.Asset(name="y", data=y_a, mock=torch.zeros(10, dtype=torch.long)),
])
dataset_b = sy.Dataset(name="mnist_b", asset_list=[
    sy.Asset(name="X", data=X_b, mock=torch.zeros(10, 784)),
    sy.Asset(name="y", data=y_b, mock=torch.zeros(10, dtype=torch.long)),
])

client_a.upload_dataset(dataset_a)
client_b.upload_dataset(dataset_b)
print("Client A:", client_a.datasets)
print("Client B:", client_b.datasets)

Uploading: y: 100%|██████████| 2/2 [00:01<00:00,  1.23it/s]


Client A: <syft.client.api.APIModule object at 0x7b53f8f22a20>
Client B: <syft.client.api.APIModule object at 0x7b53f8d14cb0>


In [3]:
X_ptr_a = client_a.datasets["mnist_a"].assets["X"]
y_ptr_a = client_a.datasets["mnist_a"].assets["y"]
X_ptr_b = client_b.datasets["mnist_b"].assets["X"]
y_ptr_b = client_b.datasets["mnist_b"].assets["y"]

print("Client A — X pointer:", X_ptr_a)
print("Client A — y pointer:", y_ptr_a)
print("Client B — X pointer:", X_ptr_b)
print("Client B — y pointer:", y_ptr_b)

Client A — X pointer: Asset(name='X', server_uid='6f86e62f1e944945bc9799a686aca2af', action_id='ddfd52f86b2042ac98b85b052a1d71cb')
Client A — y pointer: Asset(name='y', server_uid='6f86e62f1e944945bc9799a686aca2af', action_id='dcd42cfadc0f4260ab0737fa76993181')
Client B — X pointer: Asset(name='X', server_uid='11d693210feb45d18d1b3dda62140f58', action_id='a1ede5283e4449e2a8a2b1d376fc3031')
Client B — y pointer: Asset(name='y', server_uid='11d693210feb45d18d1b3dda62140f58', action_id='587d08a5e4644fc98bf43c7dde1a8a8a')


In [5]:
@sy.syft_function(
    input_policy=sy.ExactMatch(X=X_ptr_a, y=y_ptr_a),
    output_policy=sy.SingleExecutionExactOutput()
)
def train_client_a(X, y):
    import torch, torch.nn as nn, torch.optim as optim, torch.nn.functional as F
    class Net(nn.Module):
        def __init__(self):
            super().__init__()
            self.fc1 = nn.Linear(784, 128)
            self.fc2 = nn.Linear(128, 64)
            self.fc3 = nn.Linear(64, 10)
        def forward(self, x):
            return self.fc3(F.relu(self.fc2(F.relu(self.fc1(x)))))
    model = Net()
    opt = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    criterion = nn.CrossEntropyLoss()
    for epoch in range(2):
        for i in range(0, X.shape[0], 64):
            xb, yb = X[i:i+64], y[i:i+64]
            opt.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            opt.step()
    return model.state_dict()

@sy.syft_function(
    input_policy=sy.ExactMatch(X=X_ptr_b, y=y_ptr_b),
    output_policy=sy.SingleExecutionExactOutput()
)
def train_client_b(X, y):
    import torch, torch.nn as nn, torch.optim as optim, torch.nn.functional as F
    class Net(nn.Module):
        def __init__(self):
            super().__init__()
            self.fc1 = nn.Linear(784, 128)
            self.fc2 = nn.Linear(128, 64)
            self.fc3 = nn.Linear(64, 10)
        def forward(self, x):
            return self.fc3(F.relu(self.fc2(F.relu(self.fc1(x)))))
    model = Net()
    opt = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    criterion = nn.CrossEntropyLoss()
    for epoch in range(2):
        for i in range(0, X.shape[0], 64):
            xb, yb = X[i:i+64], y[i:i+64]
            opt.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            opt.step()
    return model.state_dict()

req_a = client_a.code.request_code_execution(train_client_a)
req_b = client_b.code.request_code_execution(train_client_b)
print("Request A:", req_a)
print("Request B:", req_b)

SyftSuccess: Syft function 'train_client_a' successfully created. To add a code request, please create a project using `project = syft.Project(...)`, then use command `project.create_code_request`.

SyftSuccess: Syft function 'train_client_b' successfully created. To add a code request, please create a project using `project = syft.Project(...)`, then use command `project.create_code_request`.

Request A: SyftError: Request f3f8660e17fb49afafce98539497ab09 already exists for this UserCode. Please use the existing request, or submit a new UserCode to create a new request.
Request B: SyftError: Request ac07aa9ebc5648cc85fcb8f5695f733f already exists for this UserCode. Please use the existing request, or submit a new UserCode to create a new request.


In [6]:
result_a = client_a.code.train_client_a(X=X_ptr_a, y=y_ptr_a)
result_b = client_b.code.train_client_b(X=X_ptr_b, y=y_ptr_b)

print("Result A type:", type(result_a))
print("Result B type:", type(result_b))
print("Result A:", result_a)
print("Result B:", result_b)

Result A type: <class 'syft.service.action.action_object.AnyActionObject'>
Result B type: <class 'syft.service.action.action_object.AnyActionObject'>
Result A: OrderedDict({'fc1.weight': tensor([[-0.0339, -0.0180,  0.0335,  ...,  0.0203, -0.0078,  0.0175],
        [ 0.0294,  0.0074,  0.0139,  ..., -0.0105, -0.0286,  0.0283],
        [-0.0073, -0.0068, -0.0046,  ...,  0.0044, -0.0335,  0.0287],
        ...,
        [-0.0315, -0.0084, -0.0297,  ...,  0.0042, -0.0155, -0.0377],
        [ 0.0121,  0.0140, -0.0323,  ...,  0.0286, -0.0301, -0.0305],
        [-0.0121, -0.0153,  0.0273,  ...,  0.0129,  0.0141,  0.0273]]), 'fc1.bias': tensor([ 0.0285,  0.0148,  0.0094, -0.0222, -0.0233, -0.0215,  0.0071,  0.0062,
         0.0240,  0.0178,  0.0335,  0.0211, -0.0308, -0.0305, -0.0052, -0.0300,
        -0.0302, -0.0008,  0.0010,  0.0188,  0.0070,  0.0018, -0.0286,  0.0272,
        -0.0236,  0.0205,  0.0013,  0.0407, -0.0302, -0.0004, -0.0303, -0.0070,
         0.0109, -0.0196, -0.0024, -0.0200,  0

In [7]:
#FedAvg aggregation and evaluation

import copy, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader

# Unwrap if ActionObject, otherwise use directly
weights_a = result_a if isinstance(result_a, dict) else result_a.get()
weights_b = result_b if isinstance(result_b, dict) else result_b.get()

# FedAvg – average corresponding weight tensors from both clients
def fed_avg(sd1, sd2):
    avg = copy.deepcopy(sd1)
    for k in avg:
        avg[k] = (sd1[k] + sd2[k]) / 2
    return avg

global_weights = fed_avg(weights_a, weights_b)
print("FedAvg complete. Keys:", list(global_weights.keys()))

# Same architecture as used in training
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)
    def forward(self, x):
        return self.fc3(F.relu(self.fc2(F.relu(self.fc1(x)))))

def evaluate(state_dict, dataset):
    model = Net()
    model.load_state_dict(state_dict)
    model.eval()
    loader = DataLoader(dataset, batch_size=1000, num_workers=0)
    correct = total = 0
    with torch.no_grad():
        for X_t, y_t in loader:
            preds = model(X_t.view(-1, 784)).argmax(dim=1)
            correct += (preds == y_t).sum().item()
            total += y_t.size(0)
    return correct / total

federated_acc = evaluate(global_weights, test_data)
single_acc    = evaluate(weights_a, test_data)

print(f"\nFederated model accuracy : {federated_acc:.4f} ({federated_acc*100:.2f}%)")
print(f"Single worker (Client A) : {single_acc:.4f} ({single_acc*100:.2f}%)")
print(f"Improvement              : {(federated_acc - single_acc)*100:+.2f}%")

FedAvg complete. Keys: ['fc1.weight', 'fc1.bias', 'fc2.weight', 'fc2.bias', 'fc3.weight', 'fc3.bias']

Federated model accuracy : 0.8306 (83.06%)
Single worker (Client A) : 0.9554 (95.54%)
Improvement              : -12.48%


In [8]:
#Both clients start from same seed (proper FL)

@sy.syft_function(
    input_policy=sy.ExactMatch(X=X_ptr_a, y=y_ptr_a),
    output_policy=sy.SingleExecutionExactOutput()
)
def train_seeded_a(X, y):
    import torch, torch.nn as nn, torch.optim as optim, torch.nn.functional as F
    torch.manual_seed(42)
    class Net(nn.Module):
        def __init__(self):
            super().__init__()
            self.fc1 = nn.Linear(784, 128)
            self.fc2 = nn.Linear(128, 64)
            self.fc3 = nn.Linear(64, 10)
        def forward(self, x):
            return self.fc3(F.relu(self.fc2(F.relu(self.fc1(x)))))
    model = Net()
    opt = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    criterion = nn.CrossEntropyLoss()
    for epoch in range(2):
        for i in range(0, X.shape[0], 64):
            xb, yb = X[i:i+64], y[i:i+64]
            opt.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            opt.step()
    return model.state_dict()

@sy.syft_function(
    input_policy=sy.ExactMatch(X=X_ptr_b, y=y_ptr_b),
    output_policy=sy.SingleExecutionExactOutput()
)
def train_seeded_b(X, y):
    import torch, torch.nn as nn, torch.optim as optim, torch.nn.functional as F
    torch.manual_seed(42)
    class Net(nn.Module):
        def __init__(self):
            super().__init__()
            self.fc1 = nn.Linear(784, 128)
            self.fc2 = nn.Linear(128, 64)
            self.fc3 = nn.Linear(64, 10)
        def forward(self, x):
            return self.fc3(F.relu(self.fc2(F.relu(self.fc1(x)))))
    model = Net()
    opt = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    criterion = nn.CrossEntropyLoss()
    for epoch in range(2):
        for i in range(0, X.shape[0], 64):
            xb, yb = X[i:i+64], y[i:i+64]
            opt.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            opt.step()
    return model.state_dict()

client_a.code.request_code_execution(train_seeded_a)
client_b.code.request_code_execution(train_seeded_b)

res_a2 = client_a.code.train_seeded_a(X=X_ptr_a, y=y_ptr_a)
res_b2 = client_b.code.train_seeded_b(X=X_ptr_b, y=y_ptr_b)

w_a2 = res_a2 if isinstance(res_a2, dict) else res_a2.get()
w_b2 = res_b2 if isinstance(res_b2, dict) else res_b2.get()

global_w2 = fed_avg(w_a2, w_b2)

fed_acc2    = evaluate(global_w2, test_data)
single_acc2 = evaluate(w_a2, test_data)

print(f"Federated model accuracy : {fed_acc2:.4f} ({fed_acc2*100:.2f}%)")
print(f"Single worker (Client A) : {single_acc2:.4f} ({single_acc2*100:.2f}%)")
print(f"Improvement              : {(fed_acc2 - single_acc2)*100:+.2f}%")

SyftSuccess: Syft function 'train_seeded_a' successfully created. To add a code request, please create a project using `project = syft.Project(...)`, then use command `project.create_code_request`.

SyftSuccess: Syft function 'train_seeded_b' successfully created. To add a code request, please create a project using `project = syft.Project(...)`, then use command `project.create_code_request`.

Federated model accuracy : 0.9588 (95.88%)
Single worker (Client A) : 0.9591 (95.91%)
Improvement              : -0.03%
